Lab 1a: Introduction to Artificial Neural Networks
=========

To start understanding artificial neural networks (ANNs), and how these lead to Deep Neural Networks (DNNs), it is good practice to simulate, and understand, one of the simplest (though not trivial, from a learning point of view) problems in machine learning. This is the case of the XOR (eXlusive OR) problem [XOR](https://en.wikipedia.org/wiki/Exclusive_or). This a complex, non-linear problem, requiring the use of a Multi-Layer Perceptron (MLP).

This first exercise will also introduce the fundamental concepts in Torch to create a neural network model and to carry out training. This will first use a simple Perceptron (input-output layers) and then a Multi-Layer Perceptron (input-hidden-output layers)

In the second part/handout, we will use MLPs for more compelx problems, usinhg the MNIST dataset on hadnwritten digits.


**The AND Logic Function**

Let us start first with the simpler logic AND function. The AND function takes two binary inputs, where each input can assume a value of 0 or 1 (or True/False). These two binary values constitute the input given to a neural network. The network then gives as output a single binary value representing the resultant AND logic combination of the two inputs, which is either 1 (True) or 0 (False). See the table of the possible binary combinations:

-  0, 0  =  0
-  1, 0  =  0
-  0, 1  =  0
-  1, 1  =  1


We will now try and implement this function using a network with just 1 layer after the input (no hidden layers). Our network will thus require an input layer with 2 input units, and one output layer with 1 unit.

Let's start our Torch exercise.

**Creating the input and output training data**

We first import torch, the python library used for many maths related functions. In our case we will use it to create the array of the input and output training data sets.


After defining our arrays, let's also print the values of the two arrays, to make sure all is working fine.

In [1]:
# Define input and output data
import torch
torch.manual_seed(0)  # For reproducibility
input_data = torch.tensor([[0,0], [0,1], [1,0], [1,1]], dtype=torch.float32)
output_data = torch.tensor([[0], [0], [0], [1]], dtype=torch.float32)

print(input_data)
print(output_data)

tensor([[0., 0.],
        [0., 1.],
        [1., 0.],
        [1., 1.]])
tensor([[0.],
        [0.],
        [0.],
        [1.]])


**Neural network model definition**

We now start creating our neural network model by first importing a few utilities from PyTorch.

We first import the __nn__ module from PyTorch, which provides building blocks for constructing neural networks, including layers, activation functions, and loss functions.

The second line imports the optim module, which contains optimization algorithms such as __SGD__ (Stochastic Gradient Descent) and __Adam__, used to update the network’s parameters during training.

In [2]:
# import of torch functions
import torch.nn as nn
import torch.optim as optim

Subsequently, we need to create our layers. Our simple perceptron has 2 layers: Input and output.
The input layer is simply provided by the vector of our input data, and we do not need to create an actual layer.

The output layer is of the _Dense_ type where each unit from the input is fully connected to each unit in the output layer. The layer will consist of only **1** unit to decide if the prediction is 0 (False) or 1 (True). Here we will use the __sigmoid__ (logistic) function since the output is either 0 or 1.

In [3]:
# Define the neural network class
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc = nn.Linear(2, 1)
        self.activation = nn.Sigmoid()
    
    def forward(self, x):
        x = self.fc(x)
        x = self.activation(x)
        return x

We will now compile the model, specifying the loss function (__mean_squared_error__ in this case), the optimiser (we will use __sgd__ for Stochastic Gradient Descent).

We then plot a summary of our neural network model.

In [4]:
# Initialize model
model = SimpleNN()

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Print model summary
print(model)

SimpleNN(
  (fc): Linear(in_features=2, out_features=1, bias=True)
  (activation): Sigmoid()
)


**Training the Neural Network Model**  

We are now ready to train our network model. 

### Explanation of the Training Process:
1. **Epochs and Batching:**  
   - The training loop runs for `epochs = 5000`, which determines how many times the entire dataset is processed.  
   - The `batch_size = 4` defines how many samples are used in each training step.  
   - `num_batches` determines how many iterations are needed per epoch to process the entire dataset.  

2. **Training Loop:**  
   - The outer loop iterates over the number of epochs.  
   - The inner loop processes the training data in batches of size 4, extracting input (`batch_x`) and output (`batch_y`).  
   - **Forward Pass:** The model predicts outputs for the batch using `model(batch_x)`.  
   - **Loss Computation:** The difference between predicted and actual values is calculated using `criterion(outputs, batch_y)`.  
   - **Backward Pass & Optimization:**  
     - `optimizer.zero_grad()` clears accumulated gradients from the previous step.  
     - `loss.backward()` computes gradients for each parameter.  
     - `optimizer.step()` updates the model’s parameters based on these gradients.  

3. **Monitoring Training Progress:**  
   - Every 1000 epochs, the current loss is printed to track the model’s improvement.  




In [5]:
# Training loop
epochs = 5000
batch_size = 4
num_batches = len(input_data) // batch_size

for epoch in range(epochs):
    for i in range(0, len(input_data), batch_size):
        batch_x = input_data[i:i+batch_size]
        batch_y = output_data[i:i+batch_size]
        
        # Forward pass
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Print loss every 1000 epochs
    if (epoch+1) % 1000 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [1000/5000], Loss: 0.1520
Epoch [2000/5000], Loss: 0.1276
Epoch [3000/5000], Loss: 0.1100
Epoch [4000/5000], Loss: 0.0968
Epoch [5000/5000], Loss: 0.0865


**Test of the trained network**

At the end, we can also test the performace of the trained model, with the __forward__ function.


In [6]:
# Testing the model
with torch.no_grad():
    predictions = model(input_data)
    print(predictions)

tensor([[0.1062],
        [0.3063],
        [0.2934],
        [0.6068]])


You can see from the results (e.g. accuracy) that the network is capable of learning to predict the correct output binary values for the AND problem. Thus a simple Perceptron with no hidden units can learn a simple, learning problem such as AND.

The XOR Problem
-----------------------------

Let us now try the same model architecture on the logic XOR function. The XOR function returns 1 only if its two binary inputs are different, and 0 otherwise. See the table of the possible binary combinations:

-  0, 0  =  0
-  1, 0  =  1
-  0, 1  =  1
-  1, 1  =  0

We will now define the inputs, outputs, and the model exactly as before, then compile and train our model.

In [7]:
# Define input and output data
torch.manual_seed(0)  # For reproducibility
input_data = torch.tensor([[0,0], [0,1], [1,0], [1,1]], dtype=torch.float32)
output_data = torch.tensor([[0], [1], [1], [0]], dtype=torch.float32)

print(input_data)
print(output_data)

# Define the neural network class
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc = nn.Linear(2, 1)
        self.activation = nn.Sigmoid()
    
    def forward(self, x):
        x = self.fc(x)
        x = self.activation(x)
        return x

# Initialize model
model = SimpleNN()

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Print model summary
print(model)

# Training loop
epochs = 5000
batch_size = 4
num_batches = len(input_data) // batch_size

for epoch in range(epochs):
    for i in range(0, len(input_data), batch_size):
        batch_x = input_data[i:i+batch_size]
        batch_y = output_data[i:i+batch_size]
        
        # Forward pass
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Print loss every 1000 epochs
    if (epoch+1) % 1000 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

tensor([[0., 0.],
        [0., 1.],
        [1., 0.],
        [1., 1.]])
tensor([[0.],
        [1.],
        [1.],
        [0.]])
SimpleNN(
  (fc): Linear(in_features=2, out_features=1, bias=True)
  (activation): Sigmoid()
)
Epoch [1000/5000], Loss: 0.2531
Epoch [2000/5000], Loss: 0.2518
Epoch [3000/5000], Loss: 0.2512
Epoch [4000/5000], Loss: 0.2508
Epoch [5000/5000], Loss: 0.2505


Let's look at the prediction for the XOR problem.

Has the network learned? 

In [8]:
# Testing the model
with torch.no_grad():
    predictions = model(input_data)
    print(predictions)

tensor([[0.4671],
        [0.5069],
        [0.4839],
        [0.5237]])


**Adding Hidden Layer to create a Multi-Layer Perceptron (MLP)**

We can see that our simple network has failed to learn the XOR function even though it was able to learn the AND function. This is because some problems, such as the XOR, are not linearly separable and require at least one hidden layer to be successfuly modeled. In fact, most non-trivial problems of interest are of this class.

Let us now try a more complex network with a hidden layer. The new MLP network will thus require an input layer with 2 input units, one hidden layer with some (minumum 2) units, and one output layer with 1 unit. This type of topology is called Multi-Layer Perceptron (MLP).

The output layer will be the same as before. The hiddent layer will be __Dense__, i.e. where each unit from the input is fully connected to each unit in the hidden layer. This layer will initially have __16__ hidden units. We will use the __relu__ activation function for the hidden layer neurons.

Finally, we will compile, train, and evaluate our model.


In [9]:
# Define the neural network class
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(2, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 1)
        self.activation = nn.Sigmoid()
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.activation(x)
        return x
    
# Initialize model
model = SimpleNN()

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Print model summary
print(model)

# Training loop
epochs = 5000
batch_size = 4
num_batches = len(input_data) // batch_size

for epoch in range(epochs):
    for i in range(0, len(input_data), batch_size):
        batch_x = input_data[i:i+batch_size]
        batch_y = output_data[i:i+batch_size]
        
        # Forward pass
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Print loss every 1000 epochs
    if (epoch+1) % 1000 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

SimpleNN(
  (fc1): Linear(in_features=2, out_features=16, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=16, out_features=1, bias=True)
  (activation): Sigmoid()
)
Epoch [1000/5000], Loss: 0.2260
Epoch [2000/5000], Loss: 0.1916
Epoch [3000/5000], Loss: 0.1599
Epoch [4000/5000], Loss: 0.1367
Epoch [5000/5000], Loss: 0.1167


Let's test the prediction for the XOR dataset with the additional hidden layer.


In [10]:
# Testing the model
with torch.no_grad():
    predictions = model(input_data)
    print(predictions)

tensor([[0.1737],
        [0.5210],
        [0.8102],
        [0.4135]])


Trying different (hyper)parameters
-----------------------------

We can carry out additional simulations, by exploring the role of some (hyper)parameters such as the number of hidden nodes, the learning rate, or a different activation function.
The code below is exactly the same as the one for the simlation we tried above. However, we will change the learning rate from the default value of 0.01 to 0.1.

In [11]:
# Define the neural network class
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(2, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 1)
        self.activation = nn.Sigmoid()
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.activation(x)
        return x
    
# Initialize model
model = SimpleNN()

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# Print model summary
print(model)

# Training loop
epochs = 5000
batch_size = 4
num_batches = len(input_data) // batch_size

for epoch in range(epochs):
    for i in range(0, len(input_data), batch_size):
        batch_x = input_data[i:i+batch_size]
        batch_y = output_data[i:i+batch_size]
        
        # Forward pass
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Print loss every 1000 epochs
    if (epoch+1) % 1000 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

SimpleNN(
  (fc1): Linear(in_features=2, out_features=16, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=16, out_features=1, bias=True)
  (activation): Sigmoid()
)
Epoch [1000/5000], Loss: 0.0121
Epoch [2000/5000], Loss: 0.0032
Epoch [3000/5000], Loss: 0.0017
Epoch [4000/5000], Loss: 0.0011
Epoch [5000/5000], Loss: 0.0008


We can see the the network now learns faster. The learning rate is one of the most important parameters that needs to be tuned carefully. If the learning rate is too large or too small for the problem, learning can fail.

Change the code above to try different parameters then execute it. Note how your changes affect the network performance.

Conclusions
-------------

This tutorial helped us understand the very basic concept of the implementation of a simple, multi-layer neural network, using the simple, benchmark problem and dataset of XOR. We are now ready to start exploring neural networks with more interesting training problems and datasets.

